# Churn Prevention Recommendation System

Loads the trained best model and preprocessing pipeline, then for any customer predicted to churn,
finds the **minimal actionable changes** needed to flip the prediction to "Not Churn".

**Actionable features** (only these can be changed):
- **Total Spend** → increase (offer discount / promo)
- **Contract Length** → change if it helps (Monthly → Quarterly, etc.)
- **Payment Delay** → decrease (payment reminders / auto-pay)
- **Support Calls** → decrease (proactive issue resolution)

In [2]:
import joblib
import pandas as pd
import numpy as np

# Load saved artifacts
best_model = joblib.load('../models/best_model.joblib')
pipeline = joblib.load('../models/preprocessing_pipeline.joblib')
column_info = joblib.load('../models/column_info.joblib')

feature_columns = column_info['feature_columns']
best_model_name = column_info['best_model_name']

print(f"Loaded model : {best_model_name}")
print(f"Features      : {feature_columns}")

Loaded model : Random Forest
Features      : ['Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'AgeGroup']


## Define Recommendation Engine

**Strategy — Counterfactual search with greedy minimal changes:**

1. Predict the customer. If **Not Churn** → no action needed.
2. If **Churn** → try changing **one feature at a time** (binary search for minimum).
3. Pick the single-feature change that flips the prediction with the **smallest effort**.
4. If no single feature is sufficient, combine the **top-2**, then top-3, etc. (minimized per feature).
5. **4 actionable levers:** Total Spend, Payment Delay, Support Calls, Contract Length.

In [3]:
# ---------------------------------------------------------------------------
# Thresholds & step sizes for each actionable feature
# ---------------------------------------------------------------------------
# Total Spend       — increase in steps of $50  (discount / promo)
# Payment Delay     — decrease in steps of 1 day  (payment reminders)
# Support Calls     — decrease in steps of 1 call (proactive issue resolution)
# Contract Length   — categorical: try Monthly → Quarterly → Annual (or vice versa)

NUMERIC_ACTIONS = {
    # feature name     : (direction, step_size, min_value, max_value, unit)
    'Total Spend'      : ('increase', 50,   0,  2000, '$'),
    'Payment Delay'    : ('decrease',  1,   0,    50, 'days'),
    'Support Calls'    : ('decrease',  1,   0,    15, 'calls'),
}

CONTRACT_ORDER = ['Monthly', 'Quarterly', 'Annual']  # upgrade path


def predict(row_df):
    """Return (prediction, churn_probability) for a single-row DataFrame."""
    X_proc = pipeline.transform(row_df)
    pred = best_model.predict(X_proc)[0]
    proba = best_model.predict_proba(X_proc)[0][1]
    return int(pred), float(proba)


def _try_numeric_change(base_row, feature, direction, step, min_val, max_val):
    """Binary-search for the minimal numeric nudge that flips churn to not churn.
    Returns (new_value, steps_needed) or (None, None) if impossible."""
    original = base_row[feature].values[0]

    # Check if extreme value can flip at all
    extreme = max_val if direction == 'increase' else min_val
    test = base_row.copy()
    test[feature] = extreme
    pred, _ = predict(test)
    if pred == 1:
        return None, None  # even extreme value can't flip this feature alone

    # Binary search between original and extreme
    lo, hi = 0, abs(extreme - original)
    best_delta = hi

    for _ in range(60):
        mid = (lo + hi) / 2
        candidate = original + mid if direction == 'increase' else original - mid
        test = base_row.copy()
        test[feature] = candidate
        pred, _ = predict(test)
        if pred == 0:
            best_delta = mid
            hi = mid
        else:
            lo = mid
        if hi - lo < 0.5 * step:
            break

    # Round up to nearest step
    steps_needed = int(np.ceil(best_delta / step))
    if direction == 'increase':
        new_value = original + steps_needed * step
    else:
        new_value = original - steps_needed * step
    new_value = max(min_val, min(max_val, new_value))

    # Verify rounded value actually flips
    test = base_row.copy()
    test[feature] = new_value
    pred, _ = predict(test)
    if pred == 1:
        # One extra step
        new_value = new_value + step if direction == 'increase' else new_value - step
        new_value = max(min_val, min(max_val, new_value))
        test[feature] = new_value
        pred, _ = predict(test)
        if pred == 1:
            return None, None

    return new_value, steps_needed


def _try_contract_change(base_row):
    """Try all other contract lengths. Returns (new_contract, None) or (None, None)."""
    current = base_row['Contract Length'].values[0]
    if current not in CONTRACT_ORDER:
        return None, None
    idx = CONTRACT_ORDER.index(current)
    # Try upgrade first
    for new_idx in range(idx + 1, len(CONTRACT_ORDER)):
        test = base_row.copy()
        test['Contract Length'] = CONTRACT_ORDER[new_idx]
        pred, _ = predict(test)
        if pred == 0:
            return CONTRACT_ORDER[new_idx], None
    # Then try downgrade
    for new_idx in range(idx - 1, -1, -1):
        test = base_row.copy()
        test['Contract Length'] = CONTRACT_ORDER[new_idx]
        pred, _ = predict(test)
        if pred == 0:
            return CONTRACT_ORDER[new_idx], None
    return None, None


def get_single_feature_candidates(base_row):
    """Try each actionable feature alone. Return list of viable single changes."""
    candidates = []

    for feat, (direction, step, mn, mx, unit) in NUMERIC_ACTIONS.items():
        new_val, steps = _try_numeric_change(base_row, feat, direction, step, mn, mx)
        if new_val is not None:
            original = base_row[feat].values[0]
            delta = abs(new_val - original)
            candidates.append({
                'features_changed': [feat],
                'changes': {feat: {'from': original, 'to': new_val,
                                   'delta': delta, 'unit': unit,
                                   'direction': direction}},
            })

    new_contract, _ = _try_contract_change(base_row)
    if new_contract is not None:
        original = base_row['Contract Length'].values[0]
        candidates.append({
            'features_changed': ['Contract Length'],
            'changes': {'Contract Length': {'from': original, 'to': new_contract,
                                            'delta': 0, 'unit': '',
                                            'direction': 'change'}},
        })

    return candidates


def get_multi_feature_candidates(base_row):
    """Greedy combination: find minimal feature combination to flip churn."""
    from itertools import combinations

    actionable = list(NUMERIC_ACTIONS.keys()) + ['Contract Length']

    # Best single-direction target per feature (extreme value as fallback)
    partials = {}
    for feat, (direction, step, mn, mx, unit) in NUMERIC_ACTIONS.items():
        new_val, _ = _try_numeric_change(base_row, feat, direction, step, mn, mx)
        if new_val is not None:
            partials[feat] = new_val
        else:
            # Use extreme value for combination attempts
            partials[feat] = mx if direction == 'increase' else mn

    new_contract, _ = _try_contract_change(base_row)
    if new_contract is not None:
        partials['Contract Length'] = new_contract
    elif 'Contract Length' in actionable:
        current = base_row['Contract Length'].values[0]
        idx = CONTRACT_ORDER.index(current) if current in CONTRACT_ORDER else 0
        partials['Contract Length'] = CONTRACT_ORDER[(idx + 1) % len(CONTRACT_ORDER)]

    best = None
    for r in range(2, len(actionable) + 1):
        if best is not None:
            break
        for combo in combinations(actionable, r):
            test = base_row.copy()
            changes = {}
            for feat in combo:
                if feat not in partials:
                    continue
                original = test[feat].values[0]
                test[feat] = partials[feat]
                if feat == 'Contract Length':
                    changes[feat] = {'from': original, 'to': partials[feat],
                                     'delta': 0, 'unit': '', 'direction': 'change'}
                else:
                    d, s, mn, mx, unit = NUMERIC_ACTIONS[feat]
                    changes[feat] = {'from': original, 'to': partials[feat],
                                     'delta': abs(partials[feat] - original),
                                     'unit': unit, 'direction': d}

            if len(changes) < len(combo):
                continue

            pred, _ = predict(test)
            if pred == 0:
                minimized = _minimize_combo(base_row, combo, partials)
                if minimized is not None:
                    best = {'features_changed': list(combo), 'changes': minimized}
                    break
    return best


def _minimize_combo(base_row, combo, partials):
    """Given a working combo, minimize each feature's change one at a time."""
    current = base_row.copy()
    for feat in combo:
        current[feat] = partials[feat]

    changes = {}
    for feat in combo:
        if feat == 'Contract Length':
            changes[feat] = {'from': base_row[feat].values[0], 'to': partials[feat],
                             'delta': 0, 'unit': '', 'direction': 'change'}
            continue

        direction, step, mn, mx, unit = NUMERIC_ACTIONS[feat]
        original = base_row[feat].values[0]
        target = partials[feat]

        lo, hi = 0, abs(target - original)
        best_delta = hi

        for _ in range(60):
            mid = (lo + hi) / 2
            candidate = original + mid if direction == 'increase' else original - mid
            test = current.copy()
            test[feat] = candidate
            pred, _ = predict(test)
            if pred == 0:
                best_delta = mid
                hi = mid
            else:
                lo = mid
            if hi - lo < 0.5 * step:
                break

        steps_needed = int(np.ceil(best_delta / step))
        new_val = original + steps_needed * step if direction == 'increase' else original - steps_needed * step
        new_val = max(mn, min(mx, new_val))

        # Verify
        test = current.copy()
        test[feat] = new_val
        pred, _ = predict(test)
        if pred == 1:
            new_val = new_val + step if direction == 'increase' else new_val - step
            new_val = max(mn, min(mx, new_val))

        current[feat] = new_val
        changes[feat] = {'from': original, 'to': new_val,
                         'delta': abs(new_val - original), 'unit': unit,
                         'direction': direction}

    pred, _ = predict(current)
    if pred == 1:
        return None
    return changes


def recommend(customer_row: pd.DataFrame):
    """For a single-row DataFrame, return minimal actionable recommendations to prevent churn."""
    pred, proba = predict(customer_row)

    if pred == 0:
        return {
            'status': 'NO_ACTION',
            'message': 'Customer is NOT predicted to churn. No action needed.',
            'churn_probability': proba,
        }

    # Step 1: single-feature fix
    singles = get_single_feature_candidates(customer_row)
    if singles:
        best_single = min(singles, key=lambda c: sum(v['delta'] for v in c['changes'].values()))
        test = customer_row.copy()
        for feat, chg in best_single['changes'].items():
            test[feat] = chg['to']
        new_pred, new_proba = predict(test)
        return {
            'status': 'SINGLE_CHANGE',
            'message': 'One change is enough to prevent churn.',
            'original_churn_prob': proba,
            'new_churn_prob': new_proba,
            'recommendations': best_single['changes'],
        }

    # Step 2: multi-feature combination
    multi = get_multi_feature_candidates(customer_row)
    if multi:
        test = customer_row.copy()
        for feat, chg in multi['changes'].items():
            test[feat] = chg['to']
        new_pred, new_proba = predict(test)
        return {
            'status': 'MULTI_CHANGE',
            'message': f"Combined changes across {len(multi['features_changed'])} features needed.",
            'original_churn_prob': proba,
            'new_churn_prob': new_proba,
            'recommendations': multi['changes'],
        }

    return {
        'status': 'HIGH_RISK',
        'message': 'Cannot prevent churn — escalate to retention team.',
        'churn_probability': proba,
    }


print("Recommendation engine ready.")

Recommendation engine ready.


## Pretty-Print Helper

In [4]:
def print_recommendation(result, customer_label="Customer"):
    """Nicely format the recommendation output."""
    print(f"\n{'='*65}")
    print(f"  Recommendation Report — {customer_label}")
    print(f"{'='*65}")

    if result['status'] == 'NO_ACTION':
        print(f"  ✅ {result['message']}")
        print(f"  Churn probability: {result['churn_probability']:.2%}")
        print(f"{'='*65}\n")
        return

    if result['status'] == 'HIGH_RISK':
        print(f"  🚨 {result['message']}")
        print(f"  Churn probability: {result['churn_probability']:.2%}")
        print(f"{'='*65}\n")
        return

    print(f"  ⚠️  Status: {'Single change' if result['status'] == 'SINGLE_CHANGE' else 'Multiple changes'}")
    print(f"  Churn prob BEFORE: {result['original_churn_prob']:.2%}")
    print(f"  Churn prob AFTER : {result['new_churn_prob']:.2%}")
    print(f"{'-'*65}")
    print(f"  Actionable Recommendations:")
    print(f"{'-'*65}")

    for feat, chg in result['recommendations'].items():
        if feat == 'Contract Length':
            print(f"  📋 Contract Length  : switch from '{chg['from']}' → '{chg['to']}'")
        elif feat == 'Total Spend':
            discount = chg['to'] - chg['from']
            print(f"  💰 Total Spend      : ${chg['from']:.0f} → ${chg['to']:.0f}"
                  f"  (offer ≥${discount:.0f} discount/promo)")
        elif feat == 'Payment Delay':
            print(f"  📅 Payment Delay    : {chg['from']:.0f} → {chg['to']:.0f} days"
                  f"  (reduce by {chg['delta']:.0f} days via payment reminders)")
        elif feat == 'Support Calls':
            print(f"  📞 Support Calls    : {chg['from']:.0f} → {chg['to']:.0f} calls"
                  f"  (resolve {chg['delta']:.0f} fewer escalations proactively)")

    print(f"{'='*65}\n")


print("Display helper loaded.")

Display helper loaded.


## Test: Single Customer Example

Use the same custom input from the experiment notebook to verify the recommendation engine.

In [11]:
# Example customer (from experiment notebook)
# Columns: Gender, Tenure, Usage Frequency, Support Calls, Payment Delay,
#           Subscription Type, Contract Length, Total Spend, Last Interaction, AgeGroup
custom_values = ['Female', 27, 6, 10, 29, 'Premium', 'Quarterly', 233, 10, 'Adult']
customer = pd.DataFrame([custom_values], columns=feature_columns)

print("Customer profile:")
print(customer.to_string(index=False))

pred, proba = predict(customer)
print(f"\nPrediction: {'Churn' if pred == 1 else 'Not Churn'}  (churn prob: {proba:.2%})")

result = recommend(customer)
print_recommendation(result, "Example Customer")

Customer profile:
Gender  Tenure  Usage Frequency  Support Calls  Payment Delay Subscription Type Contract Length  Total Spend  Last Interaction AgeGroup
Female      27                6             10             29           Premium       Quarterly          233                10    Adult

Prediction: Churn  (churn prob: 100.00%)

  Recommendation Report — Example Customer
  ⚠️  Status: Multiple changes
  Churn prob BEFORE: 100.00%
  Churn prob AFTER : 15.00%
-----------------------------------------------------------------
  Actionable Recommendations:
-----------------------------------------------------------------
  💰 Total Spend      : $233 → $533  (offer ≥$300 discount/promo)
  📅 Payment Delay    : 29 → 20 days  (reduce by 9 days via payment reminders)
  📞 Support Calls    : 10 → 4 calls  (resolve 6 fewer escalations proactively)

